# Лабораторная работа №7  
# Разработка автономных AI-агентов

---

**Выполнил(а):** студент(ка) группы ______  
**ФИО:** _______________________________  
**Дата:** _______________________________

---

## 1. Цель работы

- Изучить принципы создания автономных AI-агентов
- Освоить паттерны планирования действий (ReAct, Plan-and-Solve)
- Научиться создавать и использовать инструменты (tools)
- Реализовать многошаговые задачи с помощью LangGraph
- Сравнить различные архитектуры агентов

---

## 2. Теоретические сведения

### 2.1. Что такое AI-агент

**AI-агент** — это система, которая:
- Воспринимает окружение (входные данные, контекст)
- Планирует последовательность действий для достижения цели
- Использует инструменты (поиск, калькулятор, API и т.д.)
- Выполняет действия и оценивает результаты
- Корректирует план при необходимости

### 2.2. Архитектура агента

| Компонент | Назначение |
|-----------|------------|
| LLM (мозг) | Принимает решения на основе контекста |
| Планировщик | Разбивает задачу на подзадачи |
| Инструменты | Функции для выполнения конкретных действий |
| Память | Хранит историю взаимодействий |
| Исполнитель | Выполняет выбранные действия |

### 2.3. Паттерны агентов

| Паттерн | Описание | Применение |
|---------|----------|------------|
| ReAct | Reason + Act циклы | Многошаговые задачи |
| Plan-and-Solve | Сначала план, потом выполнение | Сложные вычисления |
| Multi-agent | Несколько специализированных агентов | Разделение задач |

---

## 3. Задание

### 3.1. Базовый уровень (обязательно)

1. Установите и импортируйте необходимые библиотеки (langchain, langgraph)
2. Создайте простые инструменты (калькулятор, конвертер валют)
3. Реализуйте агента с использованием ReAct паттерна
4. Протестируйте агента на простых задачах
5. Проанализируйте логи выполнения

### 3.2. Продвинутый уровень (дополнительно)

- Добавьте инструмент для поиска в интернете (SerpAPI или аналог)
- Реализуйте память агента (сохранение истории диалога)
- Создайте граф выполнения с помощью LangGraph

### 3.3. Индивидуальное задание

Разработайте агента для конкретной предметной области:
- **Вариант A:** Ассистент для анализа данных (статистика, визуализация)
- **Вариант B:** Помощник для бронирования (проверка доступности, сравнение цен)

Обоснуйте выбор инструментов и архитектуру агента.

---

## 4. Ход работы

### 4.1. Подготовка окружения

**Важно:** Запустите эту ячейку первой для установки зависимостей.

In [ ]:
# Установка зависимостей
!pip install langchain langchain-openai langgraph python-dotenv -q

# Проверка версий
import langchain
import langgraph

print(f"LangChain version: {langchain.__version__}")
print(f"LangGraph available: {langgraph is not None}")

### 4.2. Импорт библиотек и настройка API ключей

In [ ]:
import os
from google.colab import userdata

# Для Google Colab: установите API ключ в секреты (Secrets)
# Или используйте локальную переменную окружения
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except:
    os.environ["OPENAI_API_KEY"] = "your-api-key-here"  # Замените на ваш ключ

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

# Инициализация модели
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

### 4.3. Создание инструментов (Tools)

Инструменты — это функции, которые агент может вызывать для выполнения действий.

In [ ]:
@tool
def calculator(expression: str) -> str:
    """Вычисляет математическое выражение. Используйте для арифметических операций."""
    try:
        # Безопасное вычисление
        result = eval(expression, {"__builtins__": {}}, {
            "abs": abs, "round": round, "pow": pow,
            "sum": sum, "min": min, "max": max
        })
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка вычисления: {str(e)}"

@tool
def currency_converter(amount: float, from_currency: str, to_currency: str) -> str:
    """Конвертирует валюту по фиксированному курсу (пример)."""
    rates = {
        "USD": 1.0, "EUR": 0.85, "GBP": 0.73,
        "RUB": 90.0, "CNY": 7.2, "JPY": 150.0
    }
    if from_currency not in rates or to_currency not in rates:
        return "Неизвестная валюта"
    
    usd_amount = amount / rates[from_currency]
    result = usd_amount * rates[to_currency]
    return f"{amount} {from_currency} = {result:.2f} {to_currency}"

@tool
def weather_checker(city: str) -> str:
    """Проверяет погоду в городе (симуляция)."""
    import random
    conditions = ["солнечно", "облачно", "дождь", "снег"]
    temp = random.randint(-10, 35)
    condition = random.choice(conditions)
    return f"В городе {city}: {temp}°C, {condition}"

tools = [calculator, currency_converter, weather_checker]
print("Инструменты созданы:", [t.name for t in tools])

### 4.4. Создание и запуск агента

In [ ]:
# Создание ReAct агента
agent = create_react_agent(llm, tools)

# Тестовые запросы
queries = [
    "Сколько будет 25 * 4 + 100?",
    "Конвертируй 1000 рублей в доллары",
    "Какая погода в Москве?",
    "Сколько будет 150 евро в рублях?"
]

for query in queries:
    print(f"\nЗапрос: {query}")
    response = agent.invoke({"messages": [HumanMessage(content=query)]})
    print(f"Ответ: {response['messages'][-1].content}")

### 4.5. Анализ логов выполнения

In [ ]:
# Подробный вывод с историей действий
query = "Сколько будет 50 долларов в евро, а затем умножь на 2?"
print(f"Запрос: {query}\n")

response = agent.invoke({"messages": [HumanMessage(content=query)]})

print("История выполнения:")
for msg in response["messages"]:
    print(f"\n{type(msg).__name__}: {msg.content[:200] if len(msg.content) > 200 else msg.content}")

---

## 5. Контрольные вопросы

1. Какие компоненты входят в архитектуру AI-агента?
2. В чем разница между паттернами ReAct и Plan-and-Solve?
3. Как агент решает, какой инструмент использовать?
4. Какие проблемы могут возникнуть при создании агентов?
5. Как можно улучшить надежность агента?

---

## 6. Выводы

В ходе работы были изучены принципы создания автономных AI-агентов, реализованы инструменты и протестирован ReAct-агент для многошаговых задач.

---